---
title: "动态规划 (DP)——背包问题"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [18]:
#| code-fold: true
from typing import List
import math
import copy

## 📝 题目 416：分割等和子集

::: {.callout-caution}
### 📖 题目描述
给你一个 **只包含正整数** 的 **非空** 数组 `nums` 。请你判断是否可以将这个数组分割成两个子集，使得两个子集的元素和相等。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [1, 5, 11, 5]`
    * **输出**：`true`
    * **解释**：数组可以分割成 `[1, 5, 5]` 和 `[11]` 。

* **示例 2**：
    * **输入**：`nums = [1, 2, 3, 5]`
    * **输出**：`false`
    * **解释**：数组不能分割成两个元素和相等的子集。
:::

In [8]:
class Solution416:
    def canPartition(self, nums: List[int]) -> bool:
        target, mod = divmod(sum(nums), 2)  # 总和必须是偶数才能平分
        if mod != 0:
            return False
        dp = [[False] * (target + 1) for _ in range(len(nums))]  # 初始化二维 DP 数组：行数 = 物品个数，列数 = 目标容量 + 1
        for i in range(len(nums)):  # 初始化第一列：容量为 0 时，什么都不选就是 True
            dp[i][0] = True
        if nums[0] <= target: # 初始化第一行：只考虑第 0 个物品时，只有容量恰好等于它的重量时才是 True（这个物品的重量不能超过 target）
            dp[0][nums[0]] = True
        for i in range(1, len(nums)):
            for j in range(1, target + 1):
                if j < nums[i]:  # 如果当前背包容量装不下 nums[i]
                    dp[i][j] = dp[i - 1][j]
                else:  # 如果装得下
                    dp[i][j] = dp[i - 1][j] or dp[i -1][j - nums[i]]  # 选或者不选，只要有一个能凑成就是 True
        return dp[-1][-1]

In [9]:
#| code-fold: true
def test_416(func):
    test_cases = [
        {"desc": "示例 1 (正常平分)", "nums": [1, 5, 11, 5], "expected": True},
        {"desc": "示例 2 (无法平分)", "nums": [1, 2, 3, 5], "expected": False},
        {"desc": "全是偶数", "nums": [2, 2, 4, 8], "expected": True},
        {"desc": "只有一个元素", "nums": [10], "expected": False},
        {"desc": "触发越界陷阱", "nums": [100], "expected": False},
        {"desc": "触发负索引陷阱", "nums": [5, 1, 2], "expected": False}, # target=4, 算j=2,num=5时j-num<0
        {"desc": "全 1 数组 (偶数个)", "nums": [1, 1, 1, 1], "expected": True},
        {"desc": "全 1 数组 (奇数个)", "nums": [1, 1, 1, 1, 1], "expected": False},
        {"desc": "大数刚好平分", "nums": [14, 9, 8, 4, 3, 2], "expected": True}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<6} | {'实际':<6} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(list(tc["nums"]))
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<6} | {actual:<6} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_416(Solution416().canPartition)

ID   | 结果     | 预期     | 实际     | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 1      | 1      | 示例 1 (正常平分)
2    | ✅ PASS | 0      | 0      | 示例 2 (无法平分)
3    | ✅ PASS | 1      | 1      | 全是偶数
4    | ✅ PASS | 0      | 0      | 只有一个元素
5    | ✅ PASS | 0      | 0      | 触发越界陷阱
6    | ✅ PASS | 0      | 0      | 触发负索引陷阱
7    | ✅ PASS | 1      | 1      | 全 1 数组 (偶数个)
8    | ✅ PASS | 0      | 0      | 全 1 数组 (奇数个)
9    | ✅ PASS | 1      | 1      | 大数刚好平分
-----------------------------------------------------------------
测试总结: 通过 9/9


::: {.callout-important}
### 💡 思路讲解

**前期准备：**
先求出数组 `nums` 的总和 `sum`。如果 `sum` 是奇数，绝对分不开，直接 `return False`。
如果 `sum` 是偶数，我们的目标背包容量就是 `target = sum // 2`。

**动规五步曲：**

1. **确定 dp 数组以及下标的含义**：
   - 定义一个二维布尔数组 `dp[i][j]`。
   - `i` 表示：当前面对的是数组 `nums` 里的第 `i` 个数字（下标从 0 开始）。
   - `j` 表示：当前的背包容量（目标和）。
   - `dp[i][j]` 表示：**从前 `i` 个数字中随便选，能不能恰好凑出总和 `j`？**（能就是 `True`，不能就是 `False`）。

2. **确定递推公式（状态转移方程）**：
   - 现在你正看着第 `i` 个数字 `nums[i]`，你想凑出和 `j`。你只有两个选择：
   - **选择 1：不选它（或者装不下）**
     - 如果 `nums[i] > j`，这个数字比你要凑的和还要大，绝对不能选。
     - 此时 `dp[i][j]` 只能继承上一行的结果：`dp[i][j] = dp[i-1][j]`。（前面能凑出，现在就能；前面凑不出，现在也不行）。
   - **选择 2：选它**
     - 如果把它放进背包，你就得看看在**上一轮**中，能不能凑出减去它之后的容量，即 `dp[i-1][j - nums[i]]`。
   - **综合起来**：只要这两种情况有一种为 `True`，当前状态就是 `True`。
   - **方程：`dp[i][j] = dp[i-1][j] or dp[i-1][j - nums[i]]`**

3. **dp 数组如何初始化（关键！）**：
   - **第一列 (`j = 0`)**：背包容量为 0，无论面对哪个数字，只要我们“什么都不选”就能凑出 0。所以第一列全为 `True`：`dp[i][0] = True`。
   - **第一行 (`i = 0`)**：只面对第 0 个数字 `nums[0]`。它只能凑出它自己那个数值的容量。所以在第一行里，只有 `dp[0][nums[0]] = True`，其余全是 `False`。

4. **确定遍历顺序**：
   - 因为 `dp[i][j]` 依赖于上一行 `dp[i-1]` 的正上方和左上方的元素。
   - 所以我们安心地**从上到下、从左到右**遍历即可（这就是二维数组最大的好处，完全不需要像一维那样倒序绕脑子！正序遍历完全没问题！）。
   - 外层循环 `i` 从 1 遍历到 `len(nums) - 1`，内层循环 `j` 从 1 遍历到 `target`。

5. **最终结果**：
   - 填满表格后，右下角的 `dp[-1][-1]`（或者说 `dp[len(nums)-1][target]`）就是我们的最终答案。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N \times \text{target})$。其中 $N$ 是数组元素的个数，$\text{target}$ 是数组总和的一半。
* **空间复杂度**：$O(\text{target})$。我们使用了一维数组进行空间压缩。
:::

## 📝 题目 322：零钱兑换

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `coins` ，表示不同面额的硬币；以及一个整数 `amount` ，表示总金额。

计算并返回可以凑成总金额所需的 **最少的硬币个数** 。如果没有任何一种硬币组合能组成总金额，返回 `-1` 。

你可以认为每种硬币的数量是 **无限的**。

**输入输出示例**：

* **示例 1**：
    * **输入**：`coins = [1, 2, 5], amount = 11`
    * **输出**：`3`
    * **解释**：`11 = 5 + 5 + 1`

* **示例 2**：
    * **输入**：`coins = [2], amount = 3`
    * **输出**：`-1`

* **示例 3**：
    * **输入**：`coins = [1], amount = 0`
    * **输出**：`0`
:::

In [37]:
class Solution322:
    def coinChange(self, coins: List[int], amount: int) -> int:
        dp = [[math.inf] * (amount + 1) for _ in range(len(coins))]
        for i in range(len(dp)):  # 初始化第一列：凑金额 0 需要 0 个硬币
            dp[i][0] = 0
        for j in range(amount + 1):  # 初始化第一行：只用第一种硬币
            if j % coins[0] == 0:
                dp[0][j] = j // coins[0]
        for i in range(1, len(dp)):
            for j in range(1, amount + 1):
                if j < coins[i]:  # 如果目标金额小于硬币面值，说明装不下，只能不选
                    dp[i][j] = dp[i - 1][j]
                else:  # 装得下的话，在“不选（看上方）”和“选（看同行左边）”里挑最小的
                    dp[i][j] = min(dp[i - 1][j], dp[i][j - coins[i]] + 1)
        return dp[-1][-1] if dp[-1][-1] < inf else -1

    def coinChangeOptimized(self, coins: List[int], amount: int) -> int:
        dp = [math.inf] * (amount + 1)
        dp[0] = 0   # 初始化：凑金额 0 需要 0 个硬币
        for coin in coins:
            for j in range(coin, amount + 1):
                dp[j] = min(dp[j], dp[j - coin] + 1)
        return dp[-1] if dp[-1] < inf else -1

In [20]:
#| code-fold: true
def test_322(func):
    test_cases = [
        {"desc": "示例 1 (常规组合)", "coins": [1, 2, 5], "amount": 11, "expected": 3},
        {"desc": "示例 2 (无法凑出)", "coins": [2], "amount": 3, "expected": -1},
        {"desc": "示例 3 (金额为 0)", "coins": [1], "amount": 0, "expected": 0},
        {"desc": "恰好等于一个大面额", "coins": [1, 5, 10], "amount": 10, "expected": 1},
        {"desc": "需要多个大面额", "coins": [2, 5, 10, 1], "amount": 27, "expected": 4},
        {"desc": "大面额陷阱(贪心会失效)", "coins": [1, 3, 4], "amount": 6, "expected": 2}, # 贪心会选4+1+1(3个)，实际应选3+3(2个)
        {"desc": "硬币面额比金额大", "coins": [5, 10], "amount": 2, "expected": -1}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["coins"], tc["amount"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_322(Solution322().coinChange)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 3    | 3    | 示例 1 (常规组合)
2    | ✅ PASS | -1   | -1   | 示例 2 (无法凑出)
3    | ✅ PASS | 0    | 0    | 示例 3 (金额为 0)
4    | ✅ PASS | 1    | 1    | 恰好等于一个大面额
5    | ✅ PASS | 4    | 4    | 需要多个大面额
6    | ✅ PASS | 2    | 2    | 大面额陷阱(贪心会失效)
7    | ✅ PASS | -1   | -1   | 硬币面额比金额大
-----------------------------------------------------------------
测试总结: 通过 7/7


::: {.callout-important}
### 💡 思路讲解

我们要用面额为 `coins[i]` 的硬币，凑出总金额 `amount`。每种硬币都有无数个。

1. **确定 dp 数组以及下标的含义**：
   - 定义二维数组 `dp[i][j]`。
   - `i` 表示：当前只考虑前 `i` 种硬币（下标从 0 开始）。
   - `j` 表示：当前需要凑出的目标金额。
   - `dp[i][j]` 表示：**只用前 `i` 种硬币，凑出金额 `j` 所需要的最少硬币个数**。

2. **确定递推公式（🔥 见证奇迹的时刻 🔥）**：
   现在你正面对第 `i` 种硬币（面值 `coins[i]`），你要凑金额 `j`。你依然有两个选择：

   - **选择 1：不用这种硬币（或者面值太大用不了）**
     - 如果不用它，那凑出 `j` 的最少硬币数，就等于“在上一轮（不含这种硬币）凑出 `j` 的最少个数”。
     - 代码：`dp[i-1][j]`  *(和 0-1 背包一模一样)*

   - **选择 2：用这枚硬币！**
     - 如果你决定用它，你得先凑出金额 `j - coins[i]`，然后再加 1 枚当前硬币。
     - **高能预警！** 因为这种硬币有**无限个**，所以你在凑 `j - coins[i]` 的时候，**依然可以使用第 `i` 种硬币！**
     - 所以，你不是去上一行找历史数据，而是**就在当前行（第 `i` 行）**找！
     - 代码：`dp[i][j - coins[i]] + 1`  *(注意这里是 `dp[i]`，而 0-1 背包里这里是 `dp[i-1]`！)*

   - **综合起来求最小**：
     - **方程：`dp[i][j] = min(dp[i-1][j], dp[i][j - coins[i]] + 1)`**
     - *(这就是完全背包和 0-1 背包在二维宇宙里的唯一区别！)*

3. **dp 数组如何初始化**：
   - 因为我们要求“最小值”，所以除了凑金额 0 之外，其他的初始值必须是“无穷大”（可以用 `float('inf')`），否则会被 0 永远覆盖。
   - **第一列 (`j = 0`)**：凑金额 0 需要的硬币数永远是 0。`dp[i][0] = 0`。
   - **第一行 (`i = 0`)**：只用第 0 种硬币凑金额 `j`。因为必须恰好凑满，所以只有当 `j` 能被 `coins[0]` 整除时，才能凑出，个数为 `j // coins[0]`；否则依然是无穷大。

4. **填表演示 (假设 coins=[1, 2], amount=3)**

| `i` (硬币) \ `j` (金额) | 0 | 1 | 2 | 3 |
| :--- | :--- | :--- | :--- | :--- |
| **0 (硬币 1)** | 0 | 1 | 2 | 3 |
| **1 (硬币 2)** | 0 | 1 | **1** | **2** |

*带你慢动作看第二行（硬币 2）：*

* **算 `dp[1][2]`（凑金额 2）**：

  * 不用硬币2：看正上方，需要 2 个硬币（两个1）。
  * 用硬币2：看**当前行**左边金额为 0 的格子（因为 `2 - 2 = 0`），需要 0 个硬币，加上这一枚，总共 `0 + 1 = 1` 个。
  * `min(2, 1) = 1`。所以填 1。

* **算 `dp[1][3]`（凑金额 3）**：

  * 不用硬币2：看正上方，需要 3 个硬币（三个1）。
  * 用硬币2：看**当前行**左边金额为 1 的格子（因为 `3 - 2 = 1`，注意此时是在第 1 行找！），需要 1 个硬币，加上这一枚，总共 `1 + 1 = 2` 个（即一个1和一个2）。
  * `min(3, 2) = 2`。所以填 2。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N \times V)$。其中 $N$ 是硬币的种类数（len(coins)），$V$ 是总金额（amount）。我们需要填满一个 $N$ 行 $V+1$ 列的二维矩阵，每个格子的计算时间是常数级的。
* **空间复杂度**：$O(N \times V)$。我们创建了一个同样大小的二维数组 dp 来存储所有的中间状态。
:::

## 📝 题目 139：单词拆分

::: {.callout-caution}
### 📖 题目描述
给你一个字符串 `s` 和一个字符串列表 `wordDict` 作为字典。如果可以利用字典中出现的一个或多个单词拼接出 `s` 则返回 `true`。

**注意：** 不要求字典中出现的单词全部都使用，并且字典中的单词可以重复使用。

**输入输出示例**：

* **示例 1**：
    * **输入**：`s = "leetcode", wordDict = ["leet", "code"]`
    * **输出**：`true`
    * **解释**：返回 `true` 因为 `"leetcode"` 可以由 `"leet"` 和 `"code"` 拼接成。

* **示例 2**：
    * **输入**：`s = "applepenapple", wordDict = ["apple", "pen"]`
    * **输出**：`true`
    * **解释**：返回 `true` 因为 `"applepenapple"` 可以由 `"apple" + "pen" + "apple"` 拼接成。注意，你可以重复使用字典中的单词。

* **示例 3**：
    * **输入**：`s = "catsandog", wordDict = ["cats", "dog", "sand", "and", "cat"]`
    * **输出**：`false`
:::

In [31]:
class Solution139:
    def wordBreak(self, s: str, wordDict: List[str]) -> bool:
        wordSet = set(wordDict)  # 转成集合，查找更快
        dp = [False] * (len(s) + 1)
        dp[0] = True  # 初始化：0 个字符串的时候总为 True
        for i in range(len(s) + 1):
            for j in range(i, len(s) + 1):
                if dp[i] and s[i:j] in wordSet:  # 当 dp[i] 为 True 以及字符串 s[i:j] 存在时，设定下一个 dp[j] 的状态为 True
                    dp[j] = True
        return dp[-1]

In [32]:
#| code-fold: true
def test_139(func):
    test_cases = [
        {"s": "leetcode", "dict": ["leet", "code"], "exp": True, "desc": "标准两个词"},
        {"s": "applepenapple", "dict": ["apple", "pen"], "exp": True, "desc": "单词重复使用"},
        {"s": "catsandog", "dict": ["cats", "dog", "sand", "and", "cat"], "exp": False, "desc": "无法匹配末尾"},
        {"s": "apple", "dict": ["apple"], "exp": True, "desc": "整个字符串就是一个词"},
        {"s": "ab", "dict": ["a", "b"], "exp": True, "desc": "两个单字母词"},
        {"s": "aaaaaaa", "dict": ["aaaa", "aaa"], "exp": True, "desc": "重叠长度词"},
        {"s": "ccbb", "dict": ["bc", "cb"], "exp": False, "desc": "顺序错误"},
        {"s": "goalspecial", "dict": ["go", "goal", "special"], "exp": True, "desc": "前缀包含测试"},
        {"s": "bb", "dict": ["a", "b", "bbb"], "exp": True, "desc": "包含多余词"},
        {"s": "aaaaaaaaaaaaaaaaabaa", "dict": ["aa", "aaa", "aaaa"], "exp": False, "desc": "长字符串失败测试"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 45)

    for i, tc in enumerate(test_cases):
        actual = func(tc["s"], tc["dict"])
        is_pass = actual == tc["exp"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")

    print("-" * 45)
    print(f"总结: 通过 {passed}/10")

# 运行测试
test_139(Solution139().wordBreak)

ID   | 结果     | 描述
---------------------------------------------
1    | ✅ PASS | 标准两个词
2    | ✅ PASS | 单词重复使用
3    | ✅ PASS | 无法匹配末尾
4    | ✅ PASS | 整个字符串就是一个词
5    | ✅ PASS | 两个单字母词
6    | ✅ PASS | 重叠长度词
7    | ✅ PASS | 顺序错误
8    | ✅ PASS | 前缀包含测试
9    | ✅ PASS | 包含多余词
10   | ✅ PASS | 长字符串失败测试
---------------------------------------------
总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解



为什么这是完全背包？
* **背包容量**：字符串 `s` 的长度。
* **物品**：`wordDict` 里的单词（可以无限次使用）。
* **要求**：用这些单词恰好填满整个字符串，并且**顺序必须严丝合缝**。

为了解决顺序拼接的问题，我们换一种思维方式：**寻找切割点**。

1. **确定 dp 数组以及下标的含义**：
   - 定义一个一维布尔数组 `dp`，长度为 `len(s) + 1`。
   - `dp[i]` 表示：字符串的前 `i` 个字符（即 `s[0:i]`），能不能被字典里的单词拼出来。

2. **确定递推公式（状态转移方程）**：
   - 假设我们要判断前 `i` 个字符能不能拼成（求 `dp[i]`）。
   - 我们可以在 `0` 到 `i` 之间设一个切割点 `j`。把字符串分成两半：
     - 前半部分：`s[0:j]`（它的拼凑结果其实就是 `dp[j]`）
     - 后半部分：`s[j:i]`（这是一个具体的单词）
   - 只要存在**哪怕一个**切割点 `j`，使得**“前半部分能拼出来 (`dp[j] == True`)” 且 “后半部分正好在字典里 (`s[j:i] in wordDict`)”**，那么整个前 `i` 个字符就能拼出来！
   - **状态转移：`if dp[j] and s[j:i] in wordDict: dp[i] = True`**

3. **dp 数组如何初始化**：
   - `dp[0] = True`：前 0 个字符（空字符串）被认为是可以拼出的基础状态。如果没有这个 `True`，后面的状态全都会推导失败。
   - 其余的初始化为 `False`。

4. **确定遍历顺序（🔥 求排列的终极心法 🔥）**：
   - 这道题是对顺序有要求的（比如拼 `applepen`，必须先 `apple` 后 `pen`，反过来不行）。
   - **在完全背包中，如果强调顺序（求排列），必须：外层遍历背包容量（字符串长度 `i`），内层遍历物品（切割点 `j`）。**
   - 也就是说：外层循环 `i` 从 1 到 `len(s)`，代表我们当前考察的字符串长度。
   - 内层循环 `j` 从 0 到 `i-1`，代表在当前长度下，我们尝试所有的切割位置。

5. **小技巧优化**：
   - 把 `wordDict` 转换成 `set`（哈希集合），这样判断 `s[j:i] in wordDict` 的时间复杂度就能从 $O(K)$ 降到 $O(1)$！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(N^2)$。其中 $N$ 是字符串 `s` 的长度。外层循环 $N$ 次，内层循环最多 $N$ 次，每次截取字符串和哈希表查询需要 $O(1)$ 到 $O(N)$（取决于语言对切片的底层实现）。
* **空间复杂度**：$O(N)$。`dp` 数组的长度为 $N + 1$。如果将字典转为集合，还需要额外的空间存储单词。
:::